# Lesson 38 · The FFT butterfly — folding N² down to N log N with divide and conquer, and why the FFT is hundreds of times faster

**Goal**: first get the main line of Cooley–Tukey clear: compute the DFTs of the even half and the odd half separately, then use a butterfly to stitch them back into the full spectrum. Along the way, see how the cost drops from $O(N^2)$ to $O(N\log N)$.

> **Required reading order (to protect your working memory)**:
> ① just the split tree (no formulas) → ② a numeric example of one E/O butterfly pair → ③ then where $W$ comes from → ④ the cost.
> Bit reversal is an implementation detail; you can mark it "use in L39".

🔗 **Aurora connection**: `aurora.audio.transforms.fft()` is an iterative Cooley–Tukey (bit reversal + in-place butterflies). This lesson covers the *why* first; once you've implemented `butterfly()`, you'll be able to read what that `while` loop in the source is actually doing.

← **Previous lesson** · [L37 · The brute-force DFT](L37_dft.ipynb)

> Last lesson covered **the brute-force DFT**: X[k]=Σ x[n]e^{-2πikn/N}, an O(N²) double loop checked against NumPy.
> This lesson explores **the FFT butterfly**.

## This lesson's story: why is the FFT **hundreds of times** faster than the brute-force DFT?

> The factor depends on $N$: at the $N=4096$ used below it's about $N^2/( \frac{N}{2}\log_2 N) \approx 682\times$. Saying "a few hundred times" is fine — no need to memorize 200.

Imagine you need to find a particular page in a library book. The brute-force way is to turn every page — O(N). But if the book has an index, you jump to the right letter first and then search only that stretch — O(log N).

The DFT's problem is: N frequencies × N sample points = **N² multiplications**. For an audio frame of N=4096, that's 16 million operations — real-time speech processing simply can't keep up.

**The Cooley–Tukey insight** (1965):

The DFT of the even-indexed samples and the DFT of the odd-indexed samples each have only N/2 points. Merge these two "half results" with a butterfly and you get the full N-point DFT:

```
X[k]       = E[k] + W_N^k · O[k]      k = 0..N/2-1
X[k + N/2] = E[k] - W_N^k · O[k]      k = 0..N/2-1
```

This divide and conquer can **recurse**: 4096 → 2048 → 1024 → ... → 1.
Number of stages = log₂N, with N/2 multiplications per stage → **(N/2)·log₂N** in total.

At N=4096: the FFT needs about 24,576, the DFT about 16.77 million — **682 times faster**.

Your task in this section: implement `butterfly(E, O)` — the smallest testable unit of the FFT.


## 🤔 Why did engineers invent this?

- **What happens without it?** You'd use L37's O(N²) brute-force DFT directly: one 4096-point audio frame takes about 16 million multiplications. A speech stream produces a new frame every few tens of milliseconds, so in real time the CPU simply can't keep up.
- **What real problem does it solve?** Hidden inside a single DFT are lots of **half results that get computed over and over**. Divide and conquer computes each of these half results once and reuses it, folding N² down to N log N.
- **Where will it come back later?** L39 implements the whole FFT recursively, L41 adds windowing, and **the STFT in L43–L44 calls it on every single frame** — it's the foundation of the entire audio feature pipeline.

## ⚠️ Common misconceptions

> Don't think of the FFT as "a new kind of math" or "a new transform". What it computes is **exactly the same** as the DFT — the same spectrum, not one complex number different. The FFT just computes that DFT **more cleverly**: split into even and odd halves, reuse the intermediate "half results", and skip the repeated multiplications. In other words, the FFT is an **algorithmic trick**, not a new mathematical object.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

## The big-picture map: the full N=8 butterfly network

Look at the overall structure first, then read the derivation. Each colored **X** in the figure is one butterfly unit; the three colors are the three rounds of recursive merging (red → blue → green). The inputs on the left are in **bit-reversed order**, and the outputs on the right are the spectrum X[0..7] in natural order.

## 0.5 Twiddle factors: geometric intuition and where they come from

Before diving into the formulas, you need to understand one key concept: the **twiddle factor**. It isn't a magic formula — it is **forced on us** by the mathematical definition of the DFT.

### Twiddle factors: straight out of the DFT definition

Remember the DFT definition from L37?

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i kn / N}$$

Every term carries an $e^{-2\pi i kn / N}$. Now imagine we **split this sum by whether n is even or odd**:

- **Even terms** (n = 2m): $e^{-2\pi i k(2m) / N} = e^{-2\pi i km / (N/2)}$
- **Odd terms** (n = 2m+1): $e^{-2\pi i k(2m+1) / N} = e^{-2\pi i km / (N/2)} \cdot e^{-2\pi i k / N}$

See it? **The odd terms carry one extra factor, $e^{-2\pi i k / N}$, compared with the even terms**. That's where the twiddle factor comes from. Let's write it on its own:

$$W_N^k := e^{-2\pi i k / N}$$

So the $W_N^k \cdot O[k]$ in the butterfly formula is really the "rotation" that the odd terms of the original DFT carry naturally — it pops straight out of the derivation; nobody designed it out of thin air.

### Why is it called a "twiddle" (rotation) factor? — the geometry of complex multiplication

What is the complex number $e^{i\theta} = \cos\theta + i\sin\theta$ on the complex plane? It's **a point on the unit circle centered at the origin**, at angle $\theta$ from the positive real axis.

When you multiply a complex number $z$ by $e^{i\theta}$, geometrically:

$$z' = e^{i\theta} \cdot z$$

**rotates $z$ by $\theta$ radians and leaves its length unchanged** (because $|e^{i\theta}| = 1$).

In our case, $W_N^k = e^{-2\pi i k / N}$ is a **clockwise rotation by $2\pi k / N$ radians** (the minus sign means clockwise). As $k$ goes from 0 to $N-1$, the rotation angle steps from 0 toward $-2\pi$ (the last step lands at $-2\pi(N-1)/N$), going once around the origin. That's why the FFT calls it a rotation factor — in English, the "twiddle factor".

### Key property: $W_N^{k+N/2} = -W_N^k$ — why does the bottom half use a minus sign?

This is a crucial symmetry: it lets us get the bottom half of the spectrum **with no extra computation**. Let's derive it carefully.

$$W_N^{k+N/2} = e^{-2\pi i (k+N/2) / N}$$

Expand the exponent:

$$= e^{-2\pi i k / N} \cdot e^{-2\pi i (N/2) / N}$$

$$= e^{-2\pi i k / N} \cdot e^{-\pi i}$$

Now use Euler's formula $e^{i\theta} = \cos\theta + i\sin\theta$ with $\theta = -\pi$:

$$e^{-\pi i} = \cos(-\pi) + i\sin(-\pi) = -1 + 0i = -1$$

So:

$$W_N^{k+N/2} = W_N^k \cdot (-1) = -W_N^k$$

**Done.** This means the twiddle factor has a 180-degree symmetry: every $N/2$ indices, it flips sign. Using this, we can cleverly take the same set of $W_N^k$ ($k=0..N/2-1$) and, just by adding or subtracting, get the full $N$-point spectrum.



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(11, 5.5))

N = 8
n_stages = int(np.log2(N))  # = 3

xs = [0.0, 1.8, 3.6, 5.4]                  # x-coord for each column
ys = np.linspace(1.0, 0.0, N)              # y-coord: index 0 at top
COLORS = ['#e74c3c', '#3498db', '#2ecc71'] # red / blue / green per stage

# column headers
for x, lbl in zip(xs, ['Input\n(bit-reversed)', 'Stage 1\nstride 1', 'Stage 2\nstride 2', 'Stage 3\nstride 4']):
    ax.text(x, 1.12, lbl, ha='center', va='bottom', fontsize=8.5,
            fontweight='bold', color='#2c3e50')

# butterfly connections
for stage in range(n_stages):
    stride = 2 ** stage
    color  = COLORS[stage]
    x_l, x_r = xs[stage], xs[stage + 1]
    for group in range(0, N, 2 * stride):
        for k in range(stride):
            u, v   = group + k, group + k + stride
            yu, yv = ys[u], ys[v]
            ax.plot([x_l, x_r], [yu, yu], '-', color='#bdc3c7', lw=1.2, zorder=1)
            ax.plot([x_l, x_r], [yv, yv], '-', color='#bdc3c7', lw=1.2, zorder=1)
            ax.plot([x_l, x_r], [yu, yv], '-', color=color, lw=2.2, alpha=0.72, zorder=2)
            ax.plot([x_l, x_r], [yv, yu], '-', color=color, lw=2.2, alpha=0.72, zorder=2)

# nodes
for x in xs:
    for i in range(N):
        ax.plot(x, ys[i], 'o', color='white', ms=9, mec='#2c3e50', mew=1.8, zorder=5)

# input (bit-reversed) / output labels
for i, br in enumerate([0, 4, 2, 6, 1, 5, 3, 7]):
    ax.text(xs[0]  - 0.18, ys[i], f'x[{br}]', ha='right', va='center', fontsize=8.5)
    ax.text(xs[-1] + 0.18, ys[i], f'X[{i}]',  ha='left',  va='center', fontsize=8.5)

# twiddle-factor labels on last stage
x_tw = (xs[2] + xs[3]) / 2
for k in range(4):
    y_mid = (ys[k] + ys[k + 4]) / 2
    ax.text(x_tw, y_mid, f'$W_8^{k}$', ha='center', va='center',
            fontsize=8, color=COLORS[2],
            bbox=dict(fc='white', ec='none', alpha=0.85, pad=1))

# legend
ax.legend(
    handles=[mpatches.Patch(color=c, alpha=0.8, label=f'Stage {i+1}  stride {2**i}')
             for i, c in enumerate(COLORS)],
    loc='lower right', fontsize=8.5, framealpha=0.9, edgecolor='#d0d3d4',
)

ax.set_xlim(-0.95, 6.25)
ax.set_ylim(-0.08, 1.25)
ax.axis('off')
ax.set_title('N = 8  Cooley-Tukey DIT-FFT butterfly signal-flow graph\n'
             'each colored "X" is a butterfly; the three colors are the three merge stages',
             fontsize=10.5, fontweight='bold', pad=4, color='#2c3e50')
plt.tight_layout()
plt.show()

## The split, sketched: halving an N=8 DFT stage by stage down to 8 single points (build the divide-and-conquer intuition first)

For now, just keep a rough impression of the butterfly network above. Before you really read the formulas, you only need one picture: the first thing the FFT does is **cut the sequence in half by even / odd index, stage after stage**, until each piece has just 1 point (a one-point DFT is the point itself):

```text
                DFT[0..7]   (N=8)
               /            \
        even indices        odd indices
       [0 2 4 6]           [1 3 5 7]        ← split into 4 + 4
        /     \             /     \
     [0 4]   [2 6]       [1 5]   [3 7]      ← split into 2 + 2 + 2 + 2
      / \     / \         / \     / \
   [0][4] [2][6]       [1][5] [3][7]        ← down to 1, log₂8 = 3 stages in all
```

Going down is **splitting** (nearly free — just array slicing). Once you hit the bottom, you **merge back up with butterflies, stage by stage**, at N/2 multiplications per stage. log₂N stages × N/2 per stage = (N/2)·log₂N multiplications in total — that's where O(N log N) comes from. The divide-and-conquer formula below describes exactly how that "merge back up" step is computed.

### Lining them up: how does the "depth" of the recursion tree connect to the "left-to-right position" in the butterfly diagram?

The two pictures so far are drawn in completely different ways:

- **The three-colored butterfly signal-flow graph from the "big-picture map"**: drawn **left to right**, it shows the "order in which data flows" — data enters on the left, moves right, and ends up as the spectrum on the output side. The three colors (red → blue → green) run left to right.
- **The recursion tree just above**: drawn **top to bottom**, it shows the "order of calls" — first one big problem, then split into two smaller ones, then split again… all the way down to the leaves.

The two pictures actually describe **the same computation**; the axes just mean completely different things, so they're easy to mix up. The key is: **recursion "splits all the way to the bottom first, then merges from the bottom up"** — and the order in which those bottom-up merges happen is exactly the left-to-right order of the butterfly diagram.

Let's line them up concretely for N=8:

| Merge step | When it happens in the recursion | Where it sits in the butterfly diagram |
|---|---|---|
| Merge two leaves (single points) into a 2-point result, e.g. `[x0]` and `[x4]` → a 2-point DFT | Finished **first** (you must split to the deepest level before you can start coming back) | **Stage 1** (the leftmost column of butterflies, right next to the input, red, stride 1) |
| Merge two 2-point results into a 4-point result | Finished **in the middle** | **Stage 2** (the middle column, blue, stride 2) |
| Merge two 4-point results into the final 8-point spectrum | Finished **last** (the root returns last) | **Stage 3** (the rightmost column, right next to the output, green, stride 4) |

A mnemonic to remember: **"The merge the recursion finishes first is drawn furthest left; the merge it finishes last is drawn furthest right."** The reason: the signal-flow graph is drawn left to right in the order data passes through, and the data passes first through — and is first finished by — the merge closest to the leaves, which returns first.

The table alone may still feel tangled — after the "recursion demo" code cell further down, we'll have the program itself report which stage each merge belongs to, and check this correspondence against real output.

## 1. The divide-and-conquer formula: even/odd split + butterfly merge

Let `x` have length `N`, with even-indexed subsequence `x_e = x[0,2,4,...]` and odd-indexed subsequence `x_o = x[1,3,5,...]`, each of length `N/2`. Define:

```
E[k] = DFT(x_e)[k]    k = 0..N/2-1
O[k] = DFT(x_o)[k]    k = 0..N/2-1
W_N^k = exp(-2πi·k/N)       # twiddle factor
```

Then the full N-point DFT is:

```
X[k]       = E[k] + W_N^k · O[k]      k = 0..N/2-1
X[k + N/2] = E[k] - W_N^k · O[k]      k = 0..N/2-1
```

The top half `X[0..N/2-1]` adds the twiddle term, the bottom half `X[N/2..N-1]` subtracts it — because `W_N^(k+N/2) = -W_N^k`, the sign flips by itself and costs no extra computation.

In [ ]:
# Check the divide-and-conquer formula: split an N=4 signal into even/odd by hand, then merge
x = np.array([1, 2, 3, 4], dtype=complex)
N = len(x)

x_e = x[::2]   # [1, 3]
x_o = x[1::2]  # [2, 4]

E = np.fft.fft(x_e)  # 2-point DFT
O = np.fft.fft(x_o)  # 2-point DFT

k = np.arange(N // 2)
twiddles = np.exp(-2j * np.pi * k / N)  # W_N^k

X_top    = E + twiddles * O
X_bottom = E - twiddles * O
X_manual = np.concatenate([X_top, X_bottom])

X_ref = np.fft.fft(x)
print('by-hand divide & conquer:', np.round(X_manual, 4))
print('numpy.fft result        :', np.round(X_ref,    4))
print('exact match             :', np.allclose(X_manual, X_ref))

## 1.5 Where does the butterfly formula come from? — deriving it from the DFT definition

Now let's see exactly how the divide-and-conquer formula $X[k] = E[k] + W_N^k \cdot O[k]$ is derived from the original DFT definition. That way you don't have to memorize it — you'll understand where it comes from.

Given $x[0..N-1]$, the DFT definition says:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i kn / N}$$

Now **split this sum by whether n is even or odd**:

$$X[k] = \sum_{m=0}^{N/2-1} x[2m] \cdot e^{-2\pi i k(2m) / N} + \sum_{m=0}^{N/2-1} x[2m+1] \cdot e^{-2\pi i k(2m+1) / N}$$

For the first part (the even terms):

$$\sum_{m=0}^{N/2-1} x[2m] \cdot e^{-2\pi i k(2m) / N} = \sum_{m=0}^{N/2-1} x[2m] \cdot e^{-2\pi i km / (N/2)}$$

This is **exactly** $\text{DFT}(x_e)[k]$, the DFT of the even-indexed subsequence, which we call $E[k]$.

For the second part (the odd terms), substitute $n=2m+1$ into the exponent:

$$\sum_{m=0}^{N/2-1} x[2m+1] \cdot e^{-2\pi i k(2m+1) / N}$$

$$= \sum_{m=0}^{N/2-1} x[2m+1] \cdot e^{-2\pi i km / (N/2)} \cdot e^{-2\pi i k / N}$$

$$= e^{-2\pi i k / N} \sum_{m=0}^{N/2-1} x[2m+1] \cdot e^{-2\pi i km / (N/2)}$$

$$= W_N^k \cdot \text{DFT}(x_o)[k] = W_N^k \cdot O[k]$$

So:

$$X[k] = E[k] + W_N^k \cdot O[k]$$

**Nobody designed this formula out of thin air — it falls straight out of the DFT definition!** Every time you see it, you're really using the even/odd split.

Now you can also see why **the split has to be by even and odd**: for even $n$ and odd $n$, the DFT exponent $e^{-2\pi i kn/N}$ takes exactly the two forms built on $e^{-2\pi i km/(N/2)}$, and the even/odd split exploits that mathematical structure. A split that ignores this structure won't collapse into this butterfly formula.



## 2. One butterfly: 2 inputs → 2 outputs, and why the cost is O(N log N)

A **butterfly** takes a pair of complex numbers `(a, b)` and computes:

```
top    = a + W · b
bottom = a - W · b
```

It needs just 1 complex multiplication (`W·b`) and 2 complex additions/subtractions, and produces 2 outputs. Drawn on a signal flow graph, the two lines cross like a butterfly's wings — hence the name.

The stage structure of an N-point FFT:

```
stages         = log₂N
butterflies    = N/2 per stage
all butterflies = (N/2) · log₂N
```

Compare that with the naive DFT's `N²` multiplications: at `N=1024` the FFT needs only about 5120, while the naive version needs 1048576 — 200 times faster.

In [ ]:
# Compare the operation counts of the naive DFT and the FFT (theoretical values)
for n_exp in [4, 8, 10, 12]:
    N = 2 ** n_exp
    naive_mults = N * N
    fft_butterflies = (N // 2) * n_exp  # 1 multiplication per butterfly
    ratio = naive_mults / fft_butterflies
    print(f'N={N:5d}: naive={naive_mults:8d}  FFT butterflies={fft_butterflies:5d}  speedup={ratio:.1f}x')

## 3. The stopping condition: DFT(x) = x when N=1

At the very bottom of the recursion, the sequence has just 1 sample `x[0]`, and the DFT is:

```
X[0] = sum_{n=0}^{0} x[n] · exp(-2πi·0·0/1) = x[0] · 1 = x[0]
```

A one-point signal has no frequency structure at all, so the DFT output is just the input. This is the recursion's base case, and it needs no multiplications.

At each stage on the way up, two `N/2`-point results are merged by butterflies into an `N`-point result, until the top stage produces the full `X[0..N-1]`.

### The hidden assumption: N must be a power of 2

"Keep splitting down to N=1" sounds obvious — but it quietly hides a condition: **N must be a positive integer power of 2** (that is, N ∈ {1, 2, 4, 8, 16, 32, ...}).

Why is this condition needed? Think about what happens with N=6:

- First split: `x[::2]` has length 3, `x[1::2]` has length 3 — since 6 is even, this step can still cut it in half.
- But at the next stage, the length-3 sequence gets halved: `x[::2]` has length 2, `x[1::2]` has length 1 — **the two halves are no longer the same length**!

The butterfly merge `X[k] = E[k] ± W_N^k · O[k]` needs `E` and `O` to be the same length (so they can be paired up one-to-one for the additions and subtractions). As soon as the lengths differ, the recursion can't keep splitting by the same rule — either it raises an error outright, or (more dangerously) NumPy's broadcasting rules "seem to work" but produce a completely wrong result.

Every example later in the course (N=2, 4, 8) was carefully chosen to be a power of 2. That's no coincidence — it's a hard requirement of the algorithm itself. Real recordings almost never come in power-of-2 lengths (say, 44100 samples); the standard approach is **zero-padding up to the nearest power of 2** before doing the FFT, a trick covered in detail in later lessons. Below, we'll check in code exactly how N=6 goes wrong.

In [ ]:
# Recursion demo: a bare-bones hand-written recursive FFT (for teaching only, not optimized)
def fft_recursive_demo(x):
    N = len(x)
    if N == 1:
        return x.copy()  # stopping condition
    E = fft_recursive_demo(x[::2])   # even indices
    O = fft_recursive_demo(x[1::2])  # odd indices
    k = np.arange(N // 2)
    W = np.exp(-2j * np.pi * k / N)
    return np.concatenate([E + W * O, E - W * O])

x_test = np.array([1, 2, 3, 4, 5, 6, 7, 8], dtype=complex)
result = fft_recursive_demo(x_test)
print('recursive FFT:', np.round(result, 3))
print('numpy.fft    :', np.round(np.fft.fft(x_test), 3))
print('match        :', np.allclose(result, np.fft.fft(x_test)))

### Evidence: have the code report which butterfly stage each merge belongs to

Go back to the three-colored butterfly signal-flow graph in the "big-picture map" at the top. It draws three stages **left → right** (red = stage 1, stride 1; blue = stage 2, stride 2; green = stage 3, stride 4). The recursion tree in "The split, sketched" draws the splitting **top → bottom**. The two pictures use completely different coordinates and are easy to mix up — the mnemonic was:

> **The merge the recursion finishes first is drawn furthest left in the signal-flow graph; the merge it finishes last is drawn furthest right.**

Talk is cheap, so below we give `fft_recursive_demo` an "announcer": every time it finishes a butterfly merge, it reports its recursion depth, the size of the subproblem, and **which stage of the butterfly diagram this merge belongs to**. Run it and check whether it really matches the mnemonic:

In [ ]:
# Recursive FFT with an "announcer": every merge reports which butterfly stage it belongs to
def fft_trace(x, depth=0, total_depth=None):
    if total_depth is None:
        total_depth = int(np.log2(len(x)))  # the whole signal is split into log2(N) stages
    N = len(x)
    if N == 1:
        return x.copy()

    E = fft_trace(x[::2], depth + 1, total_depth)   # recurse to the bottom first (splitting)
    O = fft_trace(x[1::2], depth + 1, total_depth)
    k = np.arange(N // 2)
    W = np.exp(-2j * np.pi * k / N)

    # the larger depth is (the deeper the recursion), the earlier the merge runs → smaller stage → further left in the diagram
    stage = total_depth - depth
    stride = 2 ** (stage - 1)
    print(f'recursion depth d={depth} (subproblem N={N}) finished a merge → butterfly diagram [stage {stage}, stride {stride}]')

    return np.concatenate([E + W * O, E - W * O])

print('Order of execution (printed automatically; notice that "stage 1" always appears first and "stage 3" last):\n')
x_trace = np.arange(1, 9).astype(complex)
result_trace = fft_trace(x_trace)

print('\nmatches numpy.fft.fft:', np.allclose(result_trace, np.fft.fft(x_trace)))

### See for yourself: what happens if N isn't a power of 2?

We just said "N must be a power of 2", but reasoning alone may not convince you. Below we feed `fft_recursive_demo` a signal with N=6 (6 is even, but not a power of 2) and see whether something goes wrong:

In [ ]:
# Probe how fft_recursive_demo goes wrong with N=6 (not a power of 2)
x_bad = np.arange(1, 7).astype(complex)  # N = 6
print("x_bad =", x_bad, " N =", len(x_bad), "(6 is not a power of 2)")

try:
    fft_recursive_demo(x_bad)
    print("It actually finished? Then the result is almost certainly wrong and won't match np.fft.fft")
except ValueError as e:
    print("\n❌ Error:", e)
    print("Why: at some stage of the recursive split, the even group and the odd group have different lengths,")
    print("     and the butterfly merge E ± W·O pairs them one-to-one, so unequal lengths can't be lined up.")
    print("\nConclusion: N must be a power of 2, otherwise the recursion is bound to split unevenly at some stage.")

## Appendix B · Recursive vs iterative (production) version (just read it)

This lesson and L39 explain the FFT "recursively", because the recursion tree shows the divide-and-conquer structure most clearly:

```text
FFT(N) → FFT(N/2 even indices) + FFT(N/2 odd indices) → butterfly merge
```

But the production implementation of `aurora.audio.transforms.fft()` is an **iterative** Cooley–Tukey:

| Comparison | L39 `my_fft` | `aurora.audio.transforms.fft()` |
|---|---|---|
| Structure | recursive even/odd split | bit-reversal permutation + stage-by-stage butterflies in `while size <= n` |
| Strength | best for understanding the divide-and-conquer tree | no recursion overhead, suited to production code |
| In common | both use the same butterfly formula | both should agree numerically with `np.fft.fft` |
| Verified by | the notebook's `assert_allclose` | `tests/audio/test_transforms.py` |

You don't need to read the full source now — just know that:

- L38 teaches "how one butterfly merges"
- L39 teaches "how recursion chains many butterflies together"
- Aurora's source teaches "how real engineering flattens the recursion into an iterative loop"


## Appendix C · The bit-reversal permutation — the bridge from recursive to iterative

Why does Aurora's production code do a "bit-reversal permutation"? This part explains why the recursive and iterative versions are equivalent.

### What is the bit-reversal permutation?

The bit-reversal permutation means: **write each array index in binary, then mirror the bits end to end to get a new index**.

#### Example: bit-reversing the indices for N=8

For N=8 (3 bits are needed to write the indices 0..7):

| Original index | Binary | Bits reversed | New index |
|---|---|---|---|
| 0 | 000 | 000 | 0 |
| 1 | 001 | 100 | 4 |
| 2 | 010 | 010 | 2 |
| 3 | 011 | 110 | 6 |
| 4 | 100 | 001 | 1 |
| 5 | 101 | 101 | 5 |
| 6 | 110 | 011 | 3 |
| 7 | 111 | 111 | 7 |

So the bit-reversal permutation rearranges the original array `x[0,1,2,3,4,5,6,7]` into the order `x[0,4,2,6,1,5,3,7]`. **That's exactly the input order on the left side of the signal-flow graph earlier!**

### Why do we need bit reversal?

The recursive version of the FFT:
1. **implicitly** keeps splitting by even/odd
2. ends up with 8 single points (automatically arranged in bit-reversed order)
3. then merges back up with butterflies, stage by stage

The iterative version of the FFT:
1. **explicitly** does one bit-reversal permutation first, rearranging the input array into bit-reversed order
2. then uses a `while size <= n` loop to do the butterfly merges stage by stage (imitating the way the recursion unwinds)

Both methods compute exactly the same spectrum. The iterative version avoids the stack overhead of recursive calls and runs faster; the recursive version has clearer logic and is better for learning.

### Bit reversal in Python

If you want to see what a bit-reversal routine looks like, here is a classic loop version (Aurora's own `_bit_reverse_permutation` in `transforms.py` does the same job, vectorized with `>>` and `&`):

```python
def _bit_reverse_permutation(x, n):
    """In-place bit-reversal permutation"""
    i = 0
    for j in range(1, n):
        # step i on to the bit-reversed index of j
        k = n >> 1  # start from the top bit
        while k <= i:
            i -= k
            k >>= 1  # move one bit to the right
        i += k
        if j < i:
            x[i], x[j] = x[j], x[i]
    return x
```

This algorithm computes the bit reversal efficiently with bit operations (`>>`, the right shift). You don't need to follow every detail — just know that **"bit-reversal permutation" means reordering the array by flipping each index's binary digits**.

### Recursive ↔ iterative: the equivalence

| Recursive logic | Iterative logic |
|---|---|
| calls `fft(x[::2])` and `fft(x[1::2])` | after the bit-reversal permutation, the data is already arranged into the nested "evens first, then odds" groups |
| after the recursive calls return, `butterfly` merges | the `while size` loop unrolls the recursion, doing the butterfly merges stage by stage |
| recursion depth = log₂N, the stack is managed automatically | while-loop depth = log₂N, the loop variable is managed explicitly |
| easy to understand, easy to check | runs efficiently, suited to production code |

Once you've written the recursive FFT in L39, you'll be able to read the iterative `while size <= n` loop in Aurora's source — each pass of it corresponds to one stage of butterfly merges as the recursion unwinds.



---
⬇️ **Completion check**: you've read Appendix B; now do the N=4 half-step butterfly draft below before moving on to the main TODO.


### Warm-up: the smallest butterfly, N=2 — from two single points to a two-point DFT

Before the N=4 half-step exercise, let's look at what the **smallest butterfly** looks like.

Imagine you have two single-point signals, `x[0]` and `x[1]`. By the DFT's base case, a one-point DFT is the point itself:
- `E[0] = DFT([x[0]]) = x[0]` (the even group has only position 0)
- `O[0] = DFT([x[1]]) = x[1]` (the odd group has only position 0)

Now merge these two one-point DFTs with a butterfly to get the full two-point DFT:

$$X[0] = E[0] + W_2^0 \cdot O[0] = x[0] + 1 \cdot x[1] = x[0] + x[1]$$

$$X[1] = E[0] - W_2^0 \cdot O[0] = x[0] - 1 \cdot x[1] = x[0] - x[1]$$

where $W_2^0 = e^{-2\pi i \cdot 0 / 2} = 1$ (the twiddle factor is always 1 when k=0).

That's the most basic butterfly unit. Let's check it:



In [ ]:
# Check the smallest butterfly, N=2
x_test = np.array([3, 5], dtype=complex)  # any two complex numbers
E_n2 = x_test[0:1]  # even group: [x[0]] → its DFT is just [x[0]]
O_n2 = x_test[1:2]  # odd group: [x[1]] → its DFT is just [x[1]]

# butterfly merge
X_manual = np.array([E_n2[0] + O_n2[0], E_n2[0] - O_n2[0]], dtype=complex)
X_ref = np.fft.fft(x_test)

print(f"input: {x_test}")
print(f"E (DFT of even indices): {E_n2}")
print(f"O (DFT of odd indices): {O_n2}")
print(f"butterfly merge result: {X_manual}")
print(f"numpy.fft reference: {X_ref}")
print(f"exact match: {np.allclose(X_manual, X_ref)}")

## Half-step exercise · an N=4 butterfly draft (doesn't replace the main TODO)

Before implementing `butterfly(E, O)`, do a paper draft with N=4 first, **working out every intermediate value by hand**.

Given:

```python
x = [1, 0, 1, 0]
x_even = [1, 1]  (indices 0, 2)
x_odd  = [0, 0]  (indices 1, 3)
```

### Step 1: the DFT of the even group — E = DFT([1, 1])

Using the DFT definition $E[k] = \sum_{m=0}^{1} x_e[m] \cdot e^{-2\pi i km / 2}$ (this is an N=2 DFT):

- **k=0**: $E[0] = 1 \cdot e^{0} + 1 \cdot e^{0} = 1 + 1 = 2$
- **k=1**: $E[1] = 1 \cdot e^{0} + 1 \cdot e^{-2\pi i \cdot 1 / 2} = 1 + 1 \cdot e^{-\pi i}$
  - and $e^{-\pi i} = \cos(-\pi) + i\sin(-\pi) = -1$
  - so $E[1] = 1 + 1 \cdot (-1) = 0$

**Result**: $E = [2, 0]$

### Step 2: the DFT of the odd group — O = DFT([0, 0])

Using the same DFT definition $O[k] = \sum_{m=0}^{1} x_o[m] \cdot e^{-2\pi i km / 2}$:

- **k=0**: $O[0] = 0 \cdot e^{0} + 0 \cdot e^{0} = 0$
- **k=1**: $O[1] = 0 \cdot e^{0} + 0 \cdot e^{-\pi i} = 0$

**Result**: $O = [0, 0]$ (the DFT of an all-zero signal is, of course, all zeros)

### Step 3: the twiddle factors — $W_4^k$, k=0..1

For N=4, the twiddle factors are $W_4^k = e^{-2\pi i k / 4}$:

- **k=0**: $W_4^0 = e^{0} = 1$
- **k=1**: $W_4^1 = e^{-2\pi i / 4} = e^{-\pi i / 2}$
  - by Euler's formula: $e^{-\pi i/2} = \cos(-\pi/2) + i\sin(-\pi/2) = 0 - i = -i$

**Result**: $twiddle = [1, -i]$

### Step 4: the butterfly merge — $X[k] = E[k] \pm W_4^k \cdot O[k]$

Top half (k=0..1):
- $X[0] = E[0] + W_4^0 \cdot O[0] = 2 + 1 \cdot 0 = 2$
- $X[1] = E[1] + W_4^1 \cdot O[1] = 0 + (-i) \cdot 0 = 0$

Bottom half (k+N/2=2..3):
- $X[2] = E[0] - W_4^0 \cdot O[0] = 2 - 1 \cdot 0 = 2$
- $X[3] = E[1] - W_4^1 \cdot O[1] = 0 - (-i) \cdot 0 = 0$

**Final result**: $X = [2, 0, 2, 0]$

Now run the next cell to check your hand calculation.

In [ ]:
import numpy as np

x = np.array([1, 0, 1, 0], dtype=complex)
E = np.fft.fft(x[::2])
O = np.fft.fft(x[1::2])
k = np.arange(len(E))
twiddle = np.exp(-2j * np.pi * k / len(x))

top = E + twiddle * O
bottom = E - twiddle * O
X_half_step = np.concatenate([top, bottom])
X_ref = np.fft.fft(x)

print("E =", np.round(E, 4))
print("O =", np.round(O, 4))
print("twiddle =", np.round(twiddle, 4))
print("half-step stitched result =", np.round(X_half_step, 4))
np.testing.assert_allclose(X_half_step, X_ref, atol=1e-12)
print("✅ the N=4 half-step butterfly matches np.fft.fft exactly")


### A non-trivial example: change the numbers so neither E nor O is zero

In the hand calculation above, `O = [0, 0]` — both values happen to be zero. That's a bit too convenient, and it can make you wonder: "Did I get something wrong? Or is the odd group's DFT just always 0?"

It's not a coincidence: in `x = [1, 0, 1, 0]`, the two odd-indexed values `x[1], x[3]` are both 0 to begin with. It's a deliberately "teaching-friendly" input (tidy results, clean to check against the formula), and it doesn't represent real data. Take a signal whose four numbers are all different instead, for example:

```python
x = [1, 2, 3, 4]
x_even = [1, 3]   (indices 0, 2)
x_odd  = [2, 4]   (indices 1, 3)
```

Work it through by hand again; this time neither E nor O will be zero.

**Step 1: the DFT of the even group — E = DFT([1, 3])**

- $E[0] = 1 \cdot e^0 + 3 \cdot e^0 = 1 + 3 = 4$
- $E[1] = 1 \cdot e^0 + 3 \cdot e^{-\pi i} = 1 + 3 \cdot (-1) = 1 - 3 = -2$

**Result**: $E = [4, -2]$

**Step 2: the DFT of the odd group — O = DFT([2, 4])**

- $O[0] = 2 \cdot e^0 + 4 \cdot e^0 = 2 + 4 = 6$
- $O[1] = 2 \cdot e^0 + 4 \cdot e^{-\pi i} = 2 + 4 \cdot (-1) = 2 - 4 = -2$

**Result**: $O = [6, -2]$ (this time neither value is zero!)

**Step 3: the twiddle factors** — same as before, for N=4 $twiddle = [1, -i]$ (see step 3 above for the derivation; the values don't change, because twiddle factors depend only on N and k, not on the input values).

**Step 4: the butterfly merge — $X[k] = E[k] \pm W_4^k \cdot O[k]$**

Top half (k=0..1):
- $X[0] = E[0] + W_4^0 \cdot O[0] = 4 + 1 \cdot 6 = 10$
- $X[1] = E[1] + W_4^1 \cdot O[1] = -2 + (-i) \cdot (-2) = -2 + 2i$

Bottom half (k+N/2=2..3):
- $X[2] = E[0] - W_4^0 \cdot O[0] = 4 - 6 = -2$
- $X[3] = E[1] - W_4^1 \cdot O[1] = -2 - 2i$

**Final result**: $X = [10,\ -2+2i,\ -2,\ -2-2i]$

This time every addition, subtraction and multiplication actually works on non-zero numbers — no more "looks right but never really tested the multiplication" coincidence. Run the next cell to check your answer:

In [ ]:
x2 = np.array([1, 2, 3, 4], dtype=complex)
E2 = np.fft.fft(x2[::2])
O2 = np.fft.fft(x2[1::2])
k2 = np.arange(len(E2))
twiddle2 = np.exp(-2j * np.pi * k2 / len(x2))

top2 = E2 + twiddle2 * O2
bottom2 = E2 - twiddle2 * O2
X_nontrivial = np.concatenate([top2, bottom2])
X_ref2 = np.fft.fft(x2)

print("E =", np.round(E2, 4), " ← none of them zero")
print("O =", np.round(O2, 4), " ← none of them zero")
print("twiddle =", np.round(twiddle2, 4))
print("non-trivial example result =", np.round(X_nontrivial, 4))
np.testing.assert_allclose(X_nontrivial, X_ref2, atol=1e-12)
print("✅ the non-trivial example matches np.fft.fft exactly")

## 4. ✏️ Implement `butterfly(E, O)`

**Line of reasoning**:
1. `N = len(E) * 2`; twiddle factors `twiddles = exp(-2πi · arange(N/2) / N)`
2. `top = E + twiddles * O` (top half of the spectrum, `X[0..N/2-1]`)
3. `bottom = E - twiddles * O` (bottom half of the spectrum, `X[N/2..N-1]`)
4. return `np.concatenate([top, bottom])`, shape `(N,)`

**Reference input and output**:
```python
E = np.fft.fft(np.array([1, 3], dtype=complex))  # 2-point DFT of the even indices
O = np.fft.fft(np.array([2, 4], dtype=complex))  # 2-point DFT of the odd indices
# butterfly(E, O) ≈ np.fft.fft([1, 2, 3, 4])
# i.e. [10+0j, -2+2j, -2+0j, -2-2j]
```

> 💡 Need a hint? Once you've done the exercise, you can compare with the reference implementation in the `solutions/` folder.


In [ ]:
def butterfly(E: np.ndarray, O: np.ndarray) -> np.ndarray:
    """One stage of butterfly merging: two N/2-point DFTs → one N-point DFT.

    Parameters
    ----------
    E : shape (N/2,) complex — DFT of the even-indexed subsequence
    O : shape (N/2,) complex — DFT of the odd-indexed subsequence

    Returns
    -------
    shape (N,) complex — the full N-point DFT
    """
    half = len(E)
    N = half * 2

    # ✏️ TODO: compute the twiddle factors twiddles, shape (N/2,)
    raise NotImplementedError("TODO: compute the twiddle factors twiddles = exp(-2πi · arange(N/2) / N)")

    # ✏️ TODO: butterfly top half: top = E + twiddles * O
    raise NotImplementedError("TODO: butterfly top half top = E + twiddles * O")

    # ✏️ TODO: butterfly bottom half: bottom = E - twiddles * O
    raise NotImplementedError("TODO: butterfly bottom half bottom = E - twiddles * O")

    # ✏️ TODO: concatenate and return, shape (N,)
    raise NotImplementedError("TODO: concatenate top and bottom and return them, shape (N,)")

### Debugging example: a full N=4 butterfly, one step at a time

Before you write the code, look at a complete N=4 example that **prints every intermediate value line by line**, so you're sure you understand the vectorized operations:



In [ ]:
# Walk through the butterfly computation step by step by hand (N=4 example)
print("=" * 60)
print("N=4 butterfly, step by step: merging two N/2=2-point DFTs (even/odd) into an N=4-point DFT")
print("=" * 60)

E = np.array([2.0, 0.0], dtype=complex)  # DFT([1,1])
O = np.array([0.0, 0.0], dtype=complex)  # DFT([0,0])
N = 4

print(f"\n1️⃣  Inputs:")
print(f"   E (DFT of even indices) = {E}")
print(f"   O (DFT of odd indices) = {O}")

half = len(E)
k = np.arange(half)
print(f"\n2️⃣  Compute the twiddle factors W_4^k:")
print(f"   k array = {k}")

twiddles = np.exp(-2j * np.pi * k / N)
print(f"   W_4^k = exp(-2πi·k/4) = {np.round(twiddles, 4)}")
print(f"   In detail:")
print(f"     W_4^0 = e^0 = 1")
print(f"     W_4^1 = e^(-πi/2) = -i")

print(f"\n3️⃣  Complex multiplication W_4^k · O[k]:")
twiddles_times_O = twiddles * O
print(f"   [1, -i] × [0, 0] = {twiddles_times_O} (O is all zeros, so the result is too)")

print(f"\n4️⃣  Butterfly top half (top = E + W·O):")
top = E + twiddles_times_O
print(f"   [2, 0] + [0, 0] = {top}")

print(f"\n5️⃣  Butterfly bottom half (bottom = E - W·O):")
bottom = E - twiddles_times_O
print(f"   [2, 0] - [0, 0] = {bottom}")

print(f"\n6️⃣  Concatenate into the N-point DFT:")
X_manual = np.concatenate([top, bottom])
print(f"   X = [top | bottom] = {X_manual}")

print(f"\n✅ Reference value (np.fft.fft([1,0,1,0])):")
X_ref = np.fft.fft(np.array([1, 0, 1, 0], dtype=complex))
print(f"   {X_ref}")
print(f"\nExact match: {np.allclose(X_manual, X_ref)}")
print("=" * 60)


In [ ]:
# Check: butterfly's output should match numpy.fft.fft exactly
E = np.fft.fft(np.array([1, 3], dtype=complex))
O = np.fft.fft(np.array([2, 4], dtype=complex))
try:
    result = butterfly(E, O)
    ref = np.fft.fft(np.array([1, 2, 3, 4], dtype=complex))
    assert np.allclose(result, ref, atol=1e-10), f"mismatch: {result} vs {ref}"
    assert result.shape == (4,), f"wrong shape: expected (4,), got {result.shape}"
    print('butterfly output:', np.round(result, 4))
    print('numpy reference :', np.round(ref,    4))
    print('✅ butterfly is correct, error < 1e-10')
except (NotImplementedError, TypeError) as e:
    print(f"⬜ Implement butterfly first: {e}")


## 5. Parameter experiment: working the N=4 butterfly network by hand

An N=4 Cooley–Tukey FFT has **2 stages**, each with **2 butterflies**, for 4 twiddle-factor multiplications in all.

- **Stage 1** (the N=2 subproblems): twiddle factor `W_2^0 = 1`, so each input pair only needs an addition and a subtraction — no actual complex multiplication
- **Stage 2** (merging up to N=4): twiddle factors `W_4^0 = 1`, `W_4^1 = -i`

The naive DFT needs `4² = 16` complex multiplications; the FFT needs only `(4/2)·log₂4 = 4`.

**Experiment**: change `N` (it must be a power of 2), check whether `butterfly` combines into a full FFT, and verify the number of stages and butterflies.

---
→ **Once `butterfly` is implemented, open [L42 · The FFT drawn](L42_visual_fft.ipynb), compare the butterfly diagram with the spectra of different kinds of signals, then come back and check your `butterfly` output.**

In [ ]:
# Assemble a two-stage FFT (N=4) from butterfly and count the twiddle-factor multiplications
mult_count = [0]  # a list, so the closure can modify it

def butterfly_counted(E, O):
    """Butterfly with a counter: tallies the twiddle-factor multiplications."""
    half = len(E)
    N_local = half * 2
    k = np.arange(half)
    twiddles = np.exp(-2j * np.pi * k / N_local)
    mult_count[0] += half  # half multiplications per butterfly call
    return np.concatenate([E + twiddles * O, E - twiddles * O])

for n_exp in [2, 3, 4, 5]:
    N = 2 ** n_exp
    mult_count[0] = 0
    x = np.random.randn(N) + 1j * np.random.randn(N)

    # recursive FFT using the counting butterfly
    def fft_count(x):
        if len(x) == 1:
            return x.copy()
        E = fft_count(x[::2])
        O = fft_count(x[1::2])
        return butterfly_counted(E, O)

    result = fft_count(x)
    theory = (N // 2) * n_exp
    match = np.allclose(result, np.fft.fft(x))
    print(f'N={N:3d}: actual twiddle mults={mult_count[0]:4d}  theory (N/2·log₂N)={theory:4d}  FFT correct={match}')

## Wrap-up

`butterfly(E, O)` takes two `N/2`-point complex DFTs, merges them by adding and subtracting with the twiddle factors `W_N^k = exp(-2πi·k/N)`, and outputs the full spectrum vector of shape `(N,)`. Each call uses only `N/2` complex multiplications; unrolled over `log₂N` stages of recursion, the total cost is `O(N log N)`. This is also the algorithmic core of `aurora.audio.transforms.fft()`: the `while size <= n` loop in the source is the butterfly merging unrolled into iteration, the `even + odd`/`even - odd` in each pass corresponds to this lesson's `top`/`bottom` formulas, and `_bit_reverse_permutation` corresponds to the even/odd split order that the recursive version produces implicitly. The **recursive version** (demonstrated in this lesson) is clear but carries call-stack overhead; the **iterative version** (production code) does the bit-reversal permutation first and then unrolls in place, reusing memory with no recursion overhead, so it performs better. Next lesson (L39) implements the recursive Cooley–Tukey FFT from scratch, calling `butterfly` to complete a full N-point FFT, and verifies that its error against `np.fft.fft` is < 1e-10.

## ✏️ Closed-book derivation check cell — FFT time complexity and twiddle factors

**Rule: collapse all the cells above and do the following derivations from memory only.**

**Problem 1 — deriving the cost**: the recursive Cooley–Tukey FFT satisfies the recurrence:
$$T(N) = 2T(N/2) + O(N), \quad T(1) = O(1)$$

Unroll it over $k = \log_2 N$ stages, write out **the full sum**, and derive $T(N) = O(N \log_2 N)$.

**Reference derivation steps** (for checking your answer):

Stage 1: $T(N) = 2T(N/2) + N$

Stage 2: $T(N) = 2[2T(N/4) + N/2] + N = 4T(N/4) + 2N$

Stage 3: $T(N) = 4[2T(N/8) + N/4] + 2N = 8T(N/8) + 3N$

...

Stage i: $T(N) = 2^i T(N/2^i) + i \cdot N$

When $i = \log_2 N$, $N/2^i = 1$, so $T(1) = O(1)$.

**As a sum** (standard notation):

$$T(N) = \sum_{i=0}^{\log_2 N - 1} 2^i \cdot \frac{N}{2^i} + 2^{\log_2 N} \cdot T(1)$$

$$= \sum_{i=0}^{\log_2 N - 1} N + O(N)$$

$$= N \cdot \log_2 N + O(N)$$

$$= O(N \log N)$$

(Key observation: every stage contributes $N$ work, and there are $\log_2 N$ stages)

---

**Problem 2 — twiddle-factor table** (first stage of the N=8 butterflies)

The first stage splits $x[0..7]$ into the even group $x[0,2,4,6]$ and the odd group $x[1,3,5,7]$,
with the butterfly $X[k] = E[k] + W_8^k \cdot O[k]$.

Fill in the exact values of the twiddle factors $W_8^k = e^{-2\pi i k/8}$ for $k=0..3$:

| k | $W_8^k$ (rectangular form) |
|---|---|
| 0 | 1 |
| 1 | $\frac{\sqrt{2}}{2} - i\frac{\sqrt{2}}{2}$ |
| 2 | $-i$ |
| 3 | $-\frac{\sqrt{2}}{2} - i\frac{\sqrt{2}}{2}$ |

(If your answer matches the above, or uses decimal approximations like 0.7071, it's correct)

## Twiddle-factor guide: computing W_8^k (k=0..3) step by step

The closed-book problem asks you to fill in the twiddle-factor table by hand. This part shows you how to compute each one so you don't slip up on the signs.

### Background: Euler's formula and complex angles

Remember Euler's formula?

$$e^{i\theta} = \cos\theta + i\sin\theta$$

For a negative angle:

$$e^{-i\theta} = \cos\theta - i\sin\theta$$

(because $\cos(-\theta) = \cos\theta$ and $\sin(-\theta) = -\sin\theta$)

### Computing W_8^k

$W_8^k = e^{-2\pi i k/8} = e^{-\pi i k/4}$; for each k:

#### k=0: $W_8^0 = e^{0} = 1 + 0i$

#### k=1: $W_8^1 = e^{-\pi i/4}$

The angle is $-\pi/4 = -45°$. By Euler's formula:

$$e^{-\pi i/4} = \cos(-\pi/4) + i\sin(-\pi/4)$$

Look it up (or remember): $\cos(\pi/4) = \sin(\pi/4) = \frac{\sqrt{2}}{2}$

Because cosine is an even function and sine is odd:
- $\cos(-\pi/4) = \cos(\pi/4) = \frac{\sqrt{2}}{2}$
- $\sin(-\pi/4) = -\sin(\pi/4) = -\frac{\sqrt{2}}{2}$

So:

$$W_8^1 = \frac{\sqrt{2}}{2} - i\frac{\sqrt{2}}{2} \approx 0.7071 - 0.7071i$$

#### k=2: $W_8^2 = e^{-\pi i/2}$

The angle is $-\pi/2 = -90°$. This is a **special angle**:

$$e^{-\pi i/2} = \cos(-\pi/2) + i\sin(-\pi/2) = 0 + i(-1) = -i$$

#### k=3: $W_8^3 = e^{-3\pi i/4}$

The angle is $-3\pi/4 = -135°$. By Euler's formula:

$$e^{-3\pi i/4} = \cos(-3\pi/4) + i\sin(-3\pi/4)$$

Look it up: $\cos(3\pi/4) = -\frac{\sqrt{2}}{2}$, $\sin(3\pi/4) = \frac{\sqrt{2}}{2}$

(because $3\pi/4$ is in the second quadrant)

Because cosine is an even function and sine is odd:
- $\cos(-3\pi/4) = \cos(3\pi/4) = -\frac{\sqrt{2}}{2}$
- $\sin(-3\pi/4) = -\sin(3\pi/4) = -\frac{\sqrt{2}}{2}$

So:

$$W_8^3 = -\frac{\sqrt{2}}{2} - i\frac{\sqrt{2}}{2} \approx -0.7071 - 0.7071i$$



In [ ]:
# Check 1: the twiddle-factor table (compare your values with the formula)
import numpy as np

N = 8
ref = {k: np.exp(-2j * np.pi * k / N) for k in range(N // 2)}
print("Reference twiddle factors W_8^k (k=0..3):")
for k, w in ref.items():
    print(f"  k={k}: {w.real:+.4f} {w.imag:+.4f}j")

# Check 2: timing experiment — the slope should match N log₂ N
import sys, time
sys.path.insert(0, 'src') if 'src' not in str(sys.path) else None
try:
    from aurora.audio.transforms import fft as aurora_fft
    sizes = [128, 256, 512, 1024, 2048, 4096]
    times = []
    for n in sizes:
        x = np.random.randn(n).astype(np.float64)
        reps = max(5, 500 // n)
        t0 = time.perf_counter()
        for _ in range(reps): aurora_fft(x)
        times.append((time.perf_counter() - t0) / reps)
    expected = np.array([n * np.log2(n) for n in sizes])
    ratios = np.array(times) / expected * 1e6
    cv = ratios.std() / ratios.mean()
    assert cv < 0.6, f"slope CV={cv:.2f}, the FFT may not be O(N log N)"
    print(f"\n✅ timing check: N log N slope CV={cv:.2f} (< 0.6)")
except ImportError as e:
    print(f"⚠️ aurora.audio.transforms not found: {e}")

In [ ]:
# ✏️ Self-review for this lesson
l38_review = {
    "butterfly_formula":       None,  # remember X[k]=E[k]+W^k·O[k], X[k+N/2]=E[k]-W^k·O[k]? True/False
    "butterfly_implemented":   None,  # butterfly implemented and passing the allclose assertion? True/False
    "on2_to_onlogn":           None,  # understand why divide and conquer brings N² down to N·log N? True/False
    "twiddle_table_filled":    None,  # filled in the W_8^{0..3} twiddle-factor table closed-book? True/False
    "whiteboard_passed":       None,  # whiteboard derivation done? True/False
    "n4_half_step_done":       None,  # finished the N=4 half-step butterfly draft and understood E/O/twiddle? True/False
}

unfilled = [k for k, v in l38_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l38_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L38 fully cleared! On to L39: chaining butterflies into a full recursive FFT')


---

→ **Next lesson** · [L39 · Writing the FFT from scratch](L39_fft_implement.ipynb)

> Next lesson covers **writing the FFT from scratch**: a recursive Cooley–Tukey implementation, error < 1e-10 against numpy.fft.